In [1]:
import pandas as pd
import numpy as np
import re
import xarray as xr
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel('Dane/PositionReport.xlsx', header=[8,9])

In [3]:
czyste_kolumny = []

for col in df.columns:
    czesc_gorna = str(col[0]) if 'Unnamed' not in str(col[0]) else ''
    czesc_dolna = str(col[1]) if 'Unnamed' not in str(col[1]) else ''
    pelna_nazwa = f"{czesc_gorna} {czesc_dolna}".replace('\n', ' ').strip()
    pelna_nazwa = pelna_nazwa.replace('  ', ' ')
    czyste_kolumny.append(pelna_nazwa)
df.columns = czyste_kolumny



In [4]:

def dms_to_dd(dms_str):
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
        
    return round(dd, 5)
df.columns = df.columns.str.strip()
df['Time'] = pd.to_datetime(df['Time'], format='%d.%m.%Y %H:%M')
df['Lat_dd'] = df['Lat'].apply(dms_to_dd)
df['Lon_dd'] = df['Lon'].apply(dms_to_dd)

df_clean = df.dropna(subset=['Speed [kn]']).copy()
df_clean = df_clean[df_clean['Speed [kn]'] > 0]
print(f"Liczba wierszy przed czyszczeniem: {len(df)}")
print(f"Liczba wierszy po odcięciu postojów: {len(df_clean)}")
display(df_clean[['Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]']].head())

Liczba wierszy przed czyszczeniem: 156942
Liczba wierszy po odcięciu postojów: 132042


,Time,Lat_dd,Lon_dd,Speed [kn],Course [°]
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0


In [5]:
print("Bounding Box:")
print(f"Północ (North): {df_clean['Lat_dd'].max()}")
print(f"Południe (South): {df_clean['Lat_dd'].min()}")
print(f"Wschód (East): {df_clean['Lon_dd'].max()}")
print(f"Zachód (West): {df_clean['Lon_dd'].min()}")
print(f"Początek (Od): {df_clean['Time'].min()}")
print(f"Koniec (Do): {df_clean['Time'].max()}")

Bounding Box:
Północ (North): 61.23332
Południe (South): 51.03923
Wschód (East): 11.20088
Zachód (West): -2.12149
Początek (Od): 2023-02-02 00:06:00
Koniec (Do): 2026-02-02 08:34:57


In [6]:
df_clean.head()

,Time,Lat,Lon,Speed [kn],Calculated Speed [kn],Course [°],Distance since last point [nm],Total distance run [nm],Lat_dd,Lon_dd
0,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000,56.82667,-0.12833
1,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126,56.88667,-0.12500
2,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728,56.92333,-0.05833
3,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653,56.98667,0.05500
4,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580,57.05000,0.17500


In [22]:

# 1. PRZYGOTOWANIE CZASU I DANYCH ERA5
# Upewniamy się, że czas jest w formacie datetime bez strefy czasowej (kompatybilność z xarray)
df_clean['Time'] = pd.to_datetime(df_clean['Time'], utc=True).dt.tz_localize(None)

sciezka_do_plikow = 'Dane/*.nc'
ds_era5 = xr.open_mfdataset(sciezka_do_plikow, combine='by_coords', engine='netcdf4')

# Konwersja długości geograficznej na format -180 do 180
if ds_era5.longitude.max() > 180:
    ds_era5.coords['longitude'] = (ds_era5.coords['longitude'] + 180) % 360 - 180
    ds_era5 = ds_era5.sortby(ds_era5.longitude)

# Definicja punktów trajektorii
t = xr.DataArray(df_clean['Time'], dims="z")
lat = xr.DataArray(df_clean['Lat_dd'], dims="z")
lon = xr.DataArray(df_clean['Lon_dd'], dims="z")

# 2. INTERPOLACJA WIATRU I FAL
# Wiatr (liniowo)
ds_wind = ds_era5[['u10', 'v10']].interp(valid_time=t, latitude=lat, longitude=lon, method='linear')
ds_wind = ds_wind.compute()

# Fale (nearest + "rozmazanie" brzegów, aby uniknąć NaN przy lądzie)
ds_waves_grid = ds_era5[['swh', 'mwd']]
ds_waves_grid = ds_waves_grid.ffill(dim='longitude', limit=2).bfill(dim='longitude', limit=2)
ds_waves_grid = ds_waves_grid.ffill(dim='latitude', limit=2).bfill(dim='latitude', limit=2)

ds_waves = ds_waves_grid.interp(valid_time=t, latitude=lat, longitude=lon, method='nearest')
ds_waves = ds_waves.compute()

# 3. PRZYPISANIE DO DATAFRAME I OBLICZENIA PODSTAWOWE
df_clean['Wind_U_10m'] = ds_wind['u10'].values
df_clean['Wind_V_10m'] = ds_wind['v10'].values
df_clean['Wind_Speed_m_s'] = np.sqrt(df_clean['Wind_U_10m']**2 + df_clean['Wind_V_10m']**2)

df_clean['Wave_Height_Sig'] = ds_waves['swh'].values
df_clean['Wave_Dir_Mean'] = ds_waves['mwd'].values

# 4. PRO FILLING & INTERPOLATION (Brak NaN-ów pod model)

# Wysokość fali: interpolacja liniowa + fill 0 (brak fali tam, gdzie brak danych)
df_clean['Wave_Height_Sig'] = df_clean['Wave_Height_Sig'].interpolate(method='linear', limit=12).fillna(0)

# Kierunek fali: Interpolacja trygonometryczna (rozbicie na wektory)
dir_rad = np.deg2rad(df_clean['Wave_Dir_Mean'])
df_clean['dir_sin'] = np.sin(dir_rad)
df_clean['dir_cos'] = np.cos(dir_rad)

# Interpolacja składowych (to jest bezpieczniejsze niż interpolacja stopni)
df_clean['dir_sin'] = df_clean['dir_sin'].interpolate(method='linear', limit=12)
df_clean['dir_cos'] = df_clean['dir_cos'].interpolate(method='linear', limit=12)

# Załatanie pozostałych luk w kierunku (na początku/końcu trasy) metodą ffill/bfill
df_clean['dir_sin'] = df_clean['dir_sin'].ffill().bfill()
df_clean['dir_cos'] = df_clean['dir_cos'].ffill().bfill()

# Rekonstrukcja kierunku w stopniach (dla czytelności)
interpolated_dir_rad = np.arctan2(df_clean['dir_sin'], df_clean['dir_cos'])
df_clean['Wave_Dir_Mean'] = (np.rad2deg(interpolated_dir_rad) + 360) % 360

# 5. OPCJA PRO: ML ENCODING (Sin/Cos jako finalne cechy dla modelu)
# Te kolumny są najlepsze dla modeli typu sieci neuronowe czy XGBoost
df_clean['Wave_Dir_Sin'] = df_clean['dir_sin']
df_clean['Wave_Dir_Cos'] = df_clean['dir_cos']

# Usuwamy kolumny pomocnicze
df_clean = df_clean.drop(columns=['dir_sin', 'dir_cos'])

# Wypełnianie braków wietrznych
kolumny_wiatr = ['Wind_U_10m', 'Wind_V_10m', 'Wind_Speed_m_s']
df_clean[kolumny_wiatr] = df_clean[kolumny_wiatr].ffill().bfill()

# 6. PODSUMOWANIE
print("\n--- Połączone i Oczyszczone Dane (Gotowe pod Model) ---")
display(df_clean[['Time', 'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos','Wind_U_10m','Wind_V_10m']].head())

print("\nPodsumowanie zbioru (Liczba braków NaN):")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Wave_Dir_Sin', 'Wave_Dir_Cos']].isna().sum()
print(stats)

if stats.sum() == 0:
    print("\n✅ Sukces: Wszystkie braki danych zostały usunięte. Dane gotowe do predykcji.")


--- Połączone i Oczyszczone Dane (Gotowe pod Model) ---


,Time,Wave_Height_Sig,Wave_Dir_Mean,Wave_Dir_Sin,Wave_Dir_Cos,Wind_U_10m,Wind_V_10m
0,2023-02-02 00:06:00,2.089863,334.958862,-0.423269,0.906004,7.112340,0.726958
1,2023-02-02 00:21:00,2.152363,334.076050,-0.437178,0.899375,7.243353,0.623888
2,2023-02-02 00:33:00,2.307535,330.844055,-0.487188,0.873297,7.416633,0.372183
3,2023-02-02 00:54:00,2.307535,330.844055,-0.487188,0.873297,7.656569,-0.077478
4,2023-02-02 01:15:00,2.307535,330.844055,-0.487188,0.873297,7.587544,-0.674967



Podsumowanie zbioru (Liczba braków NaN):
Wave_Height_Sig    0
Wave_Dir_Mean      0
Wind_Speed_m_s     0
Wave_Dir_Sin       0
Wave_Dir_Cos       0
dtype: int64

✅ Sukces: Wszystkie braki danych zostały usunięte. Dane gotowe do predykcji.


In [ ]:
def dms_to_dd(dms_str):
    """Konwersja współrzędnych z formatu Stopnie Minuty (DMS) na wartości dziesiętne (DD)"""
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
    return round(dd, 5)

def calculate_awa_improved(df):
    """Obliczanie wiatru pozornego na podstawie danych ze statku i ERA5"""
    df_calc = df.copy()
    
    V_s = df_calc['Speed [kn]'] * 0.514444
    
    course_rad = np.radians(df_calc['Course [°]'])
    df_calc['u_s'] = V_s * np.sin(course_rad)
    df_calc['v_s'] = V_s * np.cos(course_rad)
    
    df_calc['u_a'] = df_calc['Wind_U_10m'] - df_calc['u_s']
    df_calc['v_a'] = df_calc['Wind_V_10m'] - df_calc['v_s']    
    
    df_calc['AWA_north'] = np.degrees(np.arctan2(df_calc['u_a'], df_calc['v_a']))   
    df_calc['AWA_relative'] = (df_calc['AWA_north'] - df_calc['Course [°]'] + 180) % 360 - 180   
    
    df_calc['Apparent_Wind_Speed_m_s'] = np.sqrt(df_calc['u_a']**2 + df_calc['v_a']**2)
    
    return df_calc
    

In [ ]:
# ==========================================
# 4. OBLICZANIE WIATRU POZORNEGO DLA ROTORA
# ==========================================
print("Obliczanie parametrów wiatru pozornego...")
df_clean = calculate_awa_improved(df_clean)


In [ ]:
# ==========================================
# 5. PODSUMOWANIE I ZAPIS
# ==========================================
print("\n--- Sprawdzenie końcowe (Brakujące wartości) ---")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'AWA_relative']].isna().sum()
print(stats.to_string())

if stats.sum() == 0:
    print("\n✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane.")
else:
    print("\n⚠️ Uwaga: Zostały puste wartości w danych, sprawdź plik wejściowy.")

output_path = 'dane_gotowe_rotor.csv'
kolumny_do_zapisu = [
    'Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]', 
    'Wind_Speed_m_s', 'Wind_U_10m', 'Wind_V_10m', 
    'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos',
    'AWA_relative', 'Apparent_Wind_Speed_m_s'
]

df_clean[kolumny_do_zapisu].to_csv(output_path, index=False)
print(f"\nZapisano gotowy zbiór ({len(df_clean)} wierszy) do pliku: {output_path}")